In [ ]:
import pandas as pd
import re

print("="*70)
print("FILTERING CDN URLS + FORCING HTTPS BALANCE")
print("="*70)

# Load dataset
print("\nLoading dataset...")
df = pd.read_csv("urls_master_real_only.csv")

print(f"Original dataset size: {len(df)}")
print(f"  Malicious: {(df['label']==1).sum()}")
print(f"  Legitimate: {(df['label']==0).sum()}")

# ============================================================
# STEP 1: REMOVE CDN URLS FROM MALWARE
# ============================================================
print("\n" + "="*70)
print("STEP 1: REMOVING CDN URLS FROM MALWARE")
print("="*70)

LEGITIMATE_CDN_PATTERNS = [
    r'github\.com/.*/(releases/download|raw)',
    r'githubusercontent\.com',
    r'sourceforge\.net/projects/.*/files',
    r'downloads\.sourceforge\.net',
    r'cdn\.discordapp\.com',
    r'media\.discordapp\.net',
    r'sharepoint\.com',
    r'onedrive\.live\.com',
    r'1drv\.ms',
    r'drive\.google\.com',
    r'docs\.google\.com/.*/(uc\?|file)',
    r'imgur\.com/download',
    r'dropbox\.com/.*\?dl=',
    r'mega\.nz',
    r'mediafire\.com',
    r'raw\.githubusercontent\.com',
    r'cdn\.jsdelivr\.net',
    r'unpkg\.com',
    r'cdnjs\.cloudflare\.com'
]

malicious_df = df[df['label'] == 1].copy()
cdn_matches = []

for pattern in LEGITIMATE_CDN_PATTERNS:
    matches = malicious_df[malicious_df['url'].str.contains(pattern, case=False, na=False, regex=True)]
    if len(matches) > 0:
        cdn_matches.append(matches)
        print(f"Found {len(matches)} URLs matching: {pattern}")

if cdn_matches:
    df_cdn_malware = pd.concat(cdn_matches).drop_duplicates()
    print(f"\nTotal CDN URLs to remove: {len(df_cdn_malware)}")
    df = df[~df.index.isin(df_cdn_malware.index)].copy()
else:
    print("No CDN URLs found")

print(f"\nAfter CDN removal:")
print(f"  Malicious: {(df['label']==1).sum()}")
print(f"  Legitimate: {(df['label']==0).sum()}")

# ============================================================
# STEP 2: CHECK HTTPS DISTRIBUTION
# ============================================================
print("\n" + "="*70)
print("STEP 2: ANALYZING HTTPS DISTRIBUTION")
print("="*70)

df['has_https'] = df['url'].str.startswith('https').astype(int)

print("\nHTTPS Distribution:")
https_stats = df.groupby('label')['has_https'].agg(['sum', 'count', 'mean'])
https_stats.columns = ['HTTPS_count', 'Total', 'HTTPS_percentage']
print(https_stats)

mal_https_pct = df[df['label']==1]['has_https'].mean()
leg_https_pct = df[df['label']==0]['has_https'].mean()

print(f"\nMalicious HTTPS: {mal_https_pct:.1%}")
print(f"Legitimate HTTPS: {leg_https_pct:.1%}")
print(f"Difference: {abs(mal_https_pct - leg_https_pct):.1%}")

# ============================================================
# STEP 3: FORCE HTTPS BALANCE IF NEEDED
# ============================================================
if abs(mal_https_pct - leg_https_pct) > 0.15:  # More than 15% difference
    print("\n" + "="*70)
    print("STEP 3: FORCING HTTPS BALANCE")
    print("="*70)
    print("⚠️  HTTPS imbalance detected! Balancing...")
    
    # Separate into 4 groups
    mal_https = df[(df['label']==1) & (df['has_https']==1)]
    mal_http = df[(df['label']==1) & (df['has_https']==0)]
    leg_https = df[(df['label']==0) & (df['has_https']==1)]
    leg_http = df[(df['label']==0) & (df['has_https']==0)]
    
    print(f"\nCurrent distribution:")
    print(f"  Malicious HTTPS: {len(mal_https)}")
    print(f"  Malicious HTTP:  {len(mal_http)}")
    print(f"  Legitimate HTTPS: {len(leg_https)}")
    print(f"  Legitimate HTTP:  {len(leg_http)}")
    
    # Target: 80% HTTPS, 20% HTTP for both classes
    target_https_ratio = 0.8
    
    # Calculate target sizes
    total_malicious = len(mal_https) + len(mal_http)
    total_legitimate = len(leg_https) + len(leg_http)
    
    # Balance malicious
    target_mal_https = int(total_malicious * target_https_ratio)
    target_mal_http = total_malicious - target_mal_https
    
    if len(mal_https) >= target_mal_https:
        mal_https_sampled = mal_https.sample(n=target_mal_https, random_state=42)
    else:
        mal_https_sampled = mal_https  # Keep all if not enough
        
    if len(mal_http) >= target_mal_http:
        mal_http_sampled = mal_http.sample(n=target_mal_http, random_state=42)
    else:
        mal_http_sampled = mal_http
    
    # Balance legitimate
    target_leg_https = int(total_legitimate * target_https_ratio)
    target_leg_http = total_legitimate - target_leg_https
    
    if len(leg_https) >= target_leg_https:
        leg_https_sampled = leg_https.sample(n=target_leg_https, random_state=42)
    else:
        leg_https_sampled = leg_https
        
    if len(leg_http) >= target_leg_http:
        leg_http_sampled = leg_http.sample(n=target_leg_http, random_state=42)
    else:
        leg_http_sampled = leg_http
    
    # Combine
    df = pd.concat([
        mal_https_sampled,
        mal_http_sampled,
        leg_https_sampled,
        leg_http_sampled
    ], ignore_index=True)
    
    # Recalculate
    mal_https_pct = df[df['label']==1]['has_https'].mean()
    leg_https_pct = df[df['label']==0]['has_https'].mean()
    
    print(f"\nAfter balancing:")
    print(f"  Malicious HTTPS: {mal_https_pct:.1%}")
    print(f"  Legitimate HTTPS: {leg_https_pct:.1%}")
    print(f"  Difference: {abs(mal_https_pct - leg_https_pct):.1%}")

else:
    print("\n✓ HTTPS distribution is already balanced (within 15%)")

# ============================================================
# STEP 4: BALANCE CLASS SIZES
# ============================================================
print("\n" + "="*70)
print("STEP 4: BALANCING CLASS SIZES")
print("="*70)

mal_count = (df['label']==1).sum()
leg_count = (df['label']==0).sum()
ratio = mal_count / leg_count if leg_count > 0 else 0

print(f"Class balance ratio: {ratio:.2f}")

if ratio < 0.8 or ratio > 1.2:
    print("Rebalancing classes...")
    
    df_malicious = df[df['label'] == 1]
    df_legitimate = df[df['label'] == 0]
    
    target_size = min(mal_count, leg_count)
    
    if mal_count > leg_count:
        df_malicious = df_malicious.sample(n=target_size, random_state=42)
    else:
        df_legitimate = df_legitimate.sample(n=target_size, random_state=42)
    
    df = pd.concat([df_malicious, df_legitimate], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ============================================================
# STEP 5: FINAL STATISTICS
# ============================================================
print("\n" + "="*70)
print("FINAL DATASET STATISTICS")
print("="*70)

df['has_ip'] = df['url'].str.contains(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', regex=True).astype(int)
df['url_length'] = df['url'].str.len()

print(f"\nTotal URLs: {len(df)}")
print(f"  Malicious: {(df['label']==1).sum()}")
print(f"  Legitimate: {(df['label']==0).sum()}")

print("\nHTTPS Distribution by Label:")
print(df.groupby(['label', 'has_https']).size().unstack(fill_value=0))

mal_https_final = df[df['label']==1]['has_https'].mean()
leg_https_final = df[df['label']==0]['has_https'].mean()
print(f"\nFinal HTTPS percentages:")
print(f"  Malicious: {mal_https_final:.1%}")
print(f"  Legitimate: {leg_https_final:.1%}")
print(f"  Difference: {abs(mal_https_final - leg_https_final):.1%}")

print("\nIP Address Distribution:")
print(df.groupby(['label', 'has_ip']).size().unstack(fill_value=0))

print("\nMean URL Length by Label:")
print(df.groupby('label')['url_length'].mean())

# Save
df_final = df.drop(columns=['has_https', 'has_ip', 'url_length'])
df_final.to_csv("urls_master_cleaned.csv", index=False)

print("\n" + "="*70)
print("✓ CLEANED & BALANCED DATASET SAVED")
print("="*70)
print("File: urls_master_cleaned.csv")
print(f"Total: {len(df_final)}")
print(f"Ready for training!")
print("\nNext: Update your training script to use 'urls_master_cleaned.csv'")

FILTERING CDN URLS + FORCING HTTPS BALANCE

Loading dataset...
Original dataset size: 17493
  Malicious: 12000
  Legitimate: 5493

STEP 1: REMOVING CDN URLS FROM MALWARE
Found 582 URLs matching: github\.com/.*/(releases/download|raw)
Found 143 URLs matching: githubusercontent\.com
Found 57 URLs matching: sharepoint\.com
Found 9 URLs matching: onedrive\.live\.com
Found 12 URLs matching: 1drv\.ms
Found 69 URLs matching: drive\.google\.com
Found 142 URLs matching: raw\.githubusercontent\.com

Total CDN URLs to remove: 872

After CDN removal:
  Malicious: 11128
  Legitimate: 5493

STEP 2: ANALYZING HTTPS DISTRIBUTION

HTTPS Distribution:
       HTTPS_count  Total  HTTPS_percentage
label                                      
0             5493   5493           1.00000
1             2272  11128           0.20417

Malicious HTTPS: 20.4%
Legitimate HTTPS: 100.0%
Difference: 79.6%

STEP 3: FORCING HTTPS BALANCE
⚠️  HTTPS imbalance detected! Balancing...

Current distribution:
  Malicious HTTPS:

C:\Users\User 1\AppData\Local\Temp\ipykernel_20116\2455976814.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = malicious_df[malicious_df['url'].str.contains(pattern, case=False, na=False, regex=True)]


In [6]:
"""
Scrape REAL HTTPS Phishing/Malware URLs Only
No generated URLs - only actual live phishing sites
"""

import pandas as pd
import requests
import time

print("="*70)
print("REAL HTTPS PHISHING & MALWARE SCRAPER")
print("="*70)

all_phishing_urls = []

# ============================================================
# 1) OPENPHISH - Live Phishing Feed
# ============================================================
print("\n[1/3] Fetching OpenPhish feed...")
try:
    r = requests.get("https://openphish.com/feed.txt", timeout=30)
    if r.status_code == 200:
        urls = r.text.strip().split('\n')
        https_urls = [u.strip() for u in urls if u.strip().startswith('https://')]
        
        print(f"  Found {len(https_urls)} total HTTPS URLs")
        
        # Remove CDN URLs
        cdn_keywords = ['github.com', 'githubusercontent', 'sharepoint', 'onedrive', 
                       'drive.google', 'docs.google', 'dropbox', 'mega.nz', 'mediafire']
        
        clean_urls = []
        for url in https_urls:
            if not any(cdn in url.lower() for cdn in cdn_keywords):
                clean_urls.append(url)
        
        print(f"  After removing CDN: {len(clean_urls)} clean HTTPS phishing URLs")
        
        for url in clean_urls[:8000]:  # Take up to 8000
            all_phishing_urls.append({
                'url': url,
                'label': 1,
                'kind': 'phishing',
                'source': 'openphish'
            })
        
        print(f"  ✓ Added {len(clean_urls[:8000])} URLs")
        if clean_urls:
            print(f"  Sample: {clean_urls[0]}")
            
except Exception as e:
    print(f"  ✗ Failed: {e}")

time.sleep(2)

# ============================================================
# 2) PHISHTANK - Verified Phishing Database
# ============================================================
print("\n[2/3] Fetching PhishTank database...")
try:
    url = "http://data.phishtank.com/data/online-valid.csv"
    print(f"  Downloading from {url}...")
    
    r = requests.get(url, timeout=60)
    
    if r.status_code == 200:
        with open('phishtank_temp.csv', 'wb') as f:
            f.write(r.content)
        
        df_phishtank = pd.read_csv('phishtank_temp.csv')
        print(f"  Loaded {len(df_phishtank)} total phishing URLs")
        
        # Get only HTTPS URLs
        if 'url' in df_phishtank.columns:
            https_phish = df_phishtank[df_phishtank['url'].str.startswith('https://', na=False)]
            
            # Remove CDN URLs
            cdn_keywords = ['github.com', 'githubusercontent', 'sharepoint', 'onedrive', 
                           'drive.google', 'docs.google', 'dropbox', 'mega.nz']
            
            clean_phish = https_phish[~https_phish['url'].str.contains('|'.join(cdn_keywords), case=False, na=False)]
            
            print(f"  Found {len(clean_phish)} clean HTTPS phishing URLs")
            
            for url in clean_phish['url'].head(8000):
                all_phishing_urls.append({
                    'url': url,
                    'label': 1,
                    'kind': 'phishing',
                    'source': 'phishtank'
                })
            
            print(f"  ✓ Added {min(len(clean_phish), 8000)} URLs")
        
        import os
        if os.path.exists('phishtank_temp.csv'):
            os.remove('phishtank_temp.csv')
        
except Exception as e:
    print(f"  ✗ Failed: {e}")
    print("  (PhishTank may be rate-limited or down)")

time.sleep(2)

# ============================================================
# 3) URLHAUS - HTTPS MALWARE (NO CDN)
# ============================================================
print("\n[3/3] Downloading URLhaus malware feed...")
try:
    # Download from URLhaus live feed
    urlhaus_url = "https://urlhaus.abuse.ch/downloads/csv_recent/"
    print(f"  Downloading from {urlhaus_url}...")
    
    r = requests.get(urlhaus_url, timeout=60)
    if r.status_code != 200:
        raise Exception(f"Failed to download: HTTP {r.status_code}")
    
    # Save temporarily
    with open('urlhaus_temp.csv', 'wb') as f:
        f.write(r.content)
    
    df_urlhaus = pd.read_csv('urlhaus_temp.csv', comment="#")
    print(f"  Loaded {len(df_urlhaus)} total URLhaus entries")
    
    # Only online/active malware
    valid_statuses = ["online", "online_download"]  # Only actually active ones
    df_malware = df_urlhaus[df_urlhaus["url_status"].isin(valid_statuses)]
    print(f"  Active malware: {len(df_malware)}")
    
    # HTTPS only
    https_malware = df_malware[df_malware['url'].str.startswith('https://', na=False)]
    print(f"  HTTPS malware: {len(https_malware)}")
    
    # Remove CDN URLs
    cdn_patterns = [
        'github.com', 'githubusercontent.com', 'sourceforge.net',
        'sharepoint.com', 'onedrive.live.com', '1drv.ms',
        'drive.google.com', 'docs.google.com', 'dropbox.com',
        'mega.nz', 'mediafire.com', 'discord'
    ]
    
    clean_malware = https_malware.copy()
    for pattern in cdn_patterns:
        clean_malware = clean_malware[~clean_malware['url'].str.contains(pattern, case=False, na=False)]
    
    print(f"  After removing CDN: {len(clean_malware)} clean HTTPS malware URLs")
    
    for url in clean_malware['url'].head(8000):
        all_phishing_urls.append({
            'url': url,
            'label': 1,
            'kind': 'malware',
            'source': 'urlhaus_https'
        })
    
    print(f"  ✓ Added {min(len(clean_malware), 8000)} URLs")
    if len(clean_malware) > 0:
        print(f"  Sample: {clean_malware['url'].iloc[0]}")
    
    # Cleanup
    import os
    if os.path.exists('urlhaus_temp.csv'):
        os.remove('urlhaus_temp.csv')
    
except Exception as e:
    print(f"  ✗ Failed: {e}")

# ============================================================
# COMBINE AND SAVE
# ============================================================
print("\n" + "="*70)
print("COMBINING RESULTS")
print("="*70)

if not all_phishing_urls:
    print("\n⚠️  WARNING: No URLs were collected!")
    print("Possible issues:")
    print("  - OpenPhish/PhishTank may be down")
    print("  - mal.csv file not found")
    print("  - Network connectivity issues")
    exit(1)

df_new_malware = pd.DataFrame(all_phishing_urls)

print(f"\nTotal new HTTPS malicious URLs: {len(df_new_malware)}")
print("\nBreakdown by source:")
print(df_new_malware['source'].value_counts())

# Verify all are HTTPS
https_count = df_new_malware['url'].str.startswith('https://').sum()
print(f"\nHTTPS percentage: {https_count/len(df_new_malware)*100:.1f}%")

# Remove duplicates
original_count = len(df_new_malware)
df_new_malware = df_new_malware.drop_duplicates(subset=['url'])
print(f"\nAfter deduplication: {len(df_new_malware)} (removed {original_count - len(df_new_malware)} duplicates)")

# Save
df_new_malware.to_csv("https_malware_supplement.csv", index=False)

print("\n" + "="*70)
print("✓ SAVED TO https_malware_supplement.csv")
print("="*70)

print(f"\nCollected {len(df_new_malware)} REAL HTTPS malicious URLs")
print("All URLs are from live phishing/malware feeds")
print("No generated/fake URLs included")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("\n1. Combine with existing dataset:")
print("   python -c \"")
print("   import pandas as pd")
print("   df_old = pd.read_csv('urls_master_real_only.csv')")
print("   df_new = pd.read_csv('https_malware_supplement.csv')")
print("   df_combined = pd.concat([df_old, df_new], ignore_index=True)")
print("   df_combined = df_combined.drop_duplicates(subset=['url'])")
print("   df_combined.to_csv('urls_master_expanded.csv', index=False)")
print("   print(f'Combined: {len(df_combined)} total URLs')")
print("   \"")
print("\n2. Run filter script on urls_master_expanded.csv")
print("\n3. Extract features and train model")

REAL HTTPS PHISHING & MALWARE SCRAPER

[1/3] Fetching OpenPhish feed...
  Found 209 total HTTPS URLs
  After removing CDN: 209 clean HTTPS phishing URLs
  ✓ Added 209 URLs
  Sample: https://click-now-appeal.page.gd/

[2/3] Fetching PhishTank database...
  Loaded 47603 total phishing URLs
  Found 38473 clean HTTPS phishing URLs
  ✓ Added 8000 URLs

[3/3] Downloading URLhaus malware feed...
  Loaded 30192 total URLhaus entries
  ✗ Failed: 'url_status'

COMBINING RESULTS

Total new HTTPS malicious URLs: 8209

Breakdown by source:
source
phishtank    8000
openphish     209
Name: count, dtype: int64

HTTPS percentage: 100.0%

After deduplication: 8208 (removed 1 duplicates)

✓ SAVED TO https_malware_supplement.csv

Collected 8208 REAL HTTPS malicious URLs
All URLs are from live phishing/malware feeds
No generated/fake URLs included

NEXT STEPS

1. Combine with existing dataset:
   python -c "
   import pandas as pd
   df_old = pd.read_csv('urls_master_real_only.csv')
   df_new = pd.read_csv

In [7]:
import pandas as pd

df = pd.read_csv("urls_master_real_only.csv")

print(f"Before: {len(df)} total")
print(f"  Malicious: {(df['label']==1).sum()}")
print(f"  Legitimate: {(df['label']==0).sum()}")

# Keep ALL legitimate (already 100% HTTPS)
df_legitimate = df[df['label'] == 0]

# Keep ONLY HTTPS malicious
df_malicious = df[df['label'] == 1]
df_malicious_https = df_malicious[df_malicious['url'].str.startswith('https://')]

print(f"\nHTTPS Malicious: {len(df_malicious_https)} (was {len(df_malicious)})")

# Combine
df_final = pd.concat([df_legitimate, df_malicious_https], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter: {len(df_final)} total")
print(f"  Malicious: {(df_final['label']==1).sum()}")
print(f"  Legitimate: {(df_final['label']==0).sum()}")

# Check HTTPS balance
mal_https = df_final[df_final['label']==1]['url'].str.startswith('https').mean()
leg_https = df_final[df_final['label']==0]['url'].str.startswith('https').mean()
print(f"\nHTTPS %: Malicious={mal_https:.1%}, Legitimate={leg_https:.1%}")

df_final.to_csv("urls_master_https_only.csv", index=False)
print("\n✓ Saved to urls_master_https_only.csv")

Before: 17493 total
  Malicious: 12000
  Legitimate: 5493

HTTPS Malicious: 3091 (was 12000)

After: 8584 total
  Malicious: 3091
  Legitimate: 5493

HTTPS %: Malicious=100.0%, Legitimate=100.0%

✓ Saved to urls_master_https_only.csv


In [ ]:
"""
Complete HTTPS Dataset Scraper
Scrapes BOTH malicious and legitimate HTTPS URLs
"""

import pandas as pd
import requests
import time
import random
import string

print("="*70)
print("COMPLETE HTTPS DATASET SCRAPER")
print("="*70)

all_malicious = []
all_legitimate = []

# ============================================================
# PART 1: MALICIOUS URLS (HTTPS ONLY)
# ============================================================
print("\n" + "="*70)
print("PART 1: SCRAPING MALICIOUS URLS (HTTPS)")
print("="*70)

# ----------------------
# 1A) OPENPHISH
# ----------------------
print("\n[1/6] Fetching OpenPhish feed...")
try:
    r = requests.get("https://openphish.com/feed.txt", timeout=30)
    if r.status_code == 200:
        urls = [u.strip() for u in r.text.strip().split('\n') if u.strip().startswith('https://')]
        
        # Remove CDN
        cdn_keywords = ['github.com', 'githubusercontent', 'sharepoint', 'onedrive', 
                       'drive.google', 'docs.google', 'dropbox', 'mega.nz', 'mediafire']
        clean_urls = [u for u in urls if not any(cdn in u.lower() for cdn in cdn_keywords)]
        
        for url in clean_urls[:8000]:
            all_malicious.append({'url': url, 'label': 1, 'kind': 'phishing', 'source': 'openphish'})
        
        print(f"  ✓ Added {len(clean_urls[:8000])} phishing URLs")
except Exception as e:
    print(f"  ✗ Failed: {e}")

time.sleep(2)

# ----------------------
# 1B) PHISHTANK
# ----------------------
print("\n[2/6] Fetching PhishTank database...")
try:
    r = requests.get("http://data.phishtank.com/data/online-valid.csv", timeout=60)
    if r.status_code == 200:
        with open('phishtank_temp.csv', 'wb') as f:
            f.write(r.content)
        
        df_pt = pd.read_csv('phishtank_temp.csv')
        if 'url' in df_pt.columns:
            https_phish = df_pt[df_pt['url'].str.startswith('https://', na=False)]
            cdn_keywords = ['github.com', 'sharepoint', 'onedrive', 'drive.google', 'docs.google']
            clean_phish = https_phish[~https_phish['url'].str.contains('|'.join(cdn_keywords), case=False, na=False)]
            
            for url in clean_phish['url'].head(8000):
                all_malicious.append({'url': url, 'label': 1, 'kind': 'phishing', 'source': 'phishtank'})
            
            print(f"  ✓ Added {min(len(clean_phish), 8000)} phishing URLs")
        
        import os
        if os.path.exists('phishtank_temp.csv'):
            os.remove('phishtank_temp.csv')
except Exception as e:
    print(f"  ✗ Failed: {e}")

time.sleep(2)

# ----------------------
# 1C) URLHAUS
# ----------------------
print("\n[3/6] Fetching URLhaus malware feed...")
try:
    r = requests.get("https://urlhaus.abuse.ch/downloads/csv_recent/", timeout=60)
    if r.status_code == 200:
        with open('urlhaus_temp.csv', 'wb') as f:
            f.write(r.content)
        
        df_uh = pd.read_csv('urlhaus_temp.csv', comment="#")
        valid_statuses = ["online", "online_download"]
        df_active = df_uh[df_uh["url_status"].isin(valid_statuses)]
        https_malware = df_active[df_active['url'].str.startswith('https://', na=False)]
        
        cdn_patterns = ['github.com', 'githubusercontent', 'sharepoint', 'onedrive', 
                       'drive.google', 'docs.google', 'dropbox', 'mega.nz', 'discord']
        clean_malware = https_malware.copy()
        for pattern in cdn_patterns:
            clean_malware = clean_malware[~clean_malware['url'].str.contains(pattern, case=False, na=False)]
        
        for url in clean_malware['url'].head(8000):
            all_malicious.append({'url': url, 'label': 1, 'kind': 'malware', 'source': 'urlhaus'})
        
        print(f"  ✓ Added {min(len(clean_malware), 8000)} malware URLs")
        
        import os
        if os.path.exists('urlhaus_temp.csv'):
            os.remove('urlhaus_temp.csv')
except Exception as e:
    print(f"  ✗ Failed: {e}")

# ============================================================
# PART 2: LEGITIMATE URLS (HTTPS ONLY)
# ============================================================
print("\n" + "="*70)
print("PART 2: SCRAPING LEGITIMATE URLS (HTTPS)")
print("="*70)

# ----------------------
# 2A) TRANCO TOP DOMAINS
# ----------------------
print("\n[4/6] Fetching Tranco top domains...")
try:
    # Download Tranco list if not exists
    try:
        df_tranco = pd.read_csv("tranco_top1m.csv", header=None, names=["rank", "domain"])
        print(f"  Loaded existing tranco_top1m.csv")
    except:
        print("  Downloading Tranco list...")
        r = requests.get("https://tranco-list.eu/download/VKGG4/1000000", timeout=60)
        with open('tranco_top1m.csv', 'wb') as f:
            f.write(r.content)
        df_tranco = pd.read_csv("tranco_top1m.csv", header=None, names=["rank", "domain"])
    
    top_domains = df_tranco["domain"].astype(str).head(10000).tolist()
    
    paths = ["", "/about", "/contact", "/blog", "/news", "/products", "/services", 
             "/help", "/support", "/search?q=example", "/category/tech"]
    
    for domain in random.sample(top_domains, 5000):
        for path in random.sample(paths, random.randint(1, 2)):
            url = f"https://{domain}{path}"
            all_legitimate.append({'url': url, 'label': 0, 'kind': 'benign', 'source': 'tranco'})
    
    print(f"  ✓ Added {len([u for u in all_legitimate if u['source']=='tranco'])} domain URLs")
    
except Exception as e:
    print(f"  ✗ Failed: {e}")

# ----------------------
# 2B) COMMON WEB SERVICES
# ----------------------
print("\n[5/6] Generating common web service URLs...")

services = []

# Google
for i in range(200):
    services.append(f"https://mail.google.com/mail/u/{i%3}/#inbox")
    services.append(f"https://drive.google.com/drive/folders/{random.randint(1000000000, 9999999999)}")
    services.append(f"https://docs.google.com/document/d/{random.randint(1000000000, 9999999999)}/edit")

# GitHub repos (NOT downloads)
popular_repos = ["microsoft/vscode", "facebook/react", "nodejs/node", "python/cpython", 
                "tensorflow/tensorflow", "pytorch/pytorch", "kubernetes/kubernetes"]
for repo in popular_repos:
    services.append(f"https://github.com/{repo}")
    services.append(f"https://github.com/{repo}/issues")
    for num in random.sample(range(1, 10000), 10):
        services.append(f"https://github.com/{repo}/issues/{num}")

# StackOverflow
for i in range(300):
    services.append(f"https://stackoverflow.com/questions/{random.randint(1000, 999999)}/how-to-solve-problem")

# Medium
for _ in range(200):
    topic = random.choice(['data-science', 'programming', 'technology', 'ai'])
    article = ''.join(random.choices(string.ascii_lowercase + string.digits, k=12))
    services.append(f"https://medium.com/{topic}/{article}")

# Reddit
subreddits = ["programming", "python", "javascript", "datascience", "technology"]
for sub in subreddits:
    services.append(f"https://www.reddit.com/r/{sub}/")
    for _ in range(30):
        tid = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
        services.append(f"https://www.reddit.com/r/{sub}/comments/{tid}/discussion/")

# YouTube
for _ in range(200):
    vid = ''.join(random.choices(string.ascii_letters + string.digits, k=11))
    services.append(f"https://www.youtube.com/watch?v={vid}")

# NPM/PyPI
packages = ["express", "react", "vue", "numpy", "pandas", "tensorflow", "flask"]
for pkg in packages:
    services.append(f"https://www.npmjs.com/package/{pkg}")
    services.append(f"https://pypi.org/project/{pkg}/")
    for _ in range(5):
        ver = f"{random.randint(1,5)}.{random.randint(0,20)}.{random.randint(0,10)}"
        services.append(f"https://www.npmjs.com/package/{pkg}/v/{ver}")

# E-commerce
for _ in range(200):
    search = random.choice(["laptop", "phone", "headphones", "keyboard"])
    services.append(f"https://www.amazon.com/s?k={search}")
    services.append(f"https://www.amazon.com/dp/B0{random.randint(10000000, 99999999)}")

for url in services:
    all_legitimate.append({'url': url, 'label': 0, 'kind': 'benign', 'source': 'web_services'})

print(f"  ✓ Added {len(services)} web service URLs")

# ----------------------
# 2C) WIKIPEDIA
# ----------------------
print("\n[6/6] Generating Wikipedia URLs...")

wiki_topics = [
    "Machine_learning", "Artificial_intelligence", "Python_(programming_language)",
    "JavaScript", "Computer_science", "Data_science", "Web_development",
    "Software_engineering", "Database", "Algorithm", "Operating_system",
    "Cybersecurity", "Cloud_computing", "Docker_(software)", "Kubernetes",
    "Git", "Stack_Overflow", "Reddit", "Google", "Microsoft", "Apple_Inc.",
    "Mathematics", "Physics", "Chemistry", "Biology", "History"
]

wiki_urls = []
for topic in wiki_topics:
    wiki_urls.append(f"https://en.wikipedia.org/wiki/{topic}")
    wiki_urls.append(f"https://en.wikipedia.org/wiki/{topic}#History")

for _ in range(500):
    words = random.sample(["Technology", "Science", "Computer", "Software", "Network"], 2)
    wiki_urls.append(f"https://en.wikipedia.org/wiki/{'_'.join(words)}")

for url in wiki_urls:
    all_legitimate.append({'url': url, 'label': 0, 'kind': 'benign', 'source': 'wikipedia'})

print(f"  ✓ Added {len(wiki_urls)} Wikipedia URLs")

# ============================================================
# COMBINE AND SAVE
# ============================================================
print("\n" + "="*70)
print("COMBINING RESULTS")
print("="*70)

df_malicious = pd.DataFrame(all_malicious)
df_legitimate = pd.DataFrame(all_legitimate)

print(f"\nMalicious URLs collected: {len(df_malicious)}")
print("  Sources:", df_malicious['source'].value_counts().to_dict())

print(f"\nLegitimate URLs collected: {len(df_legitimate)}")
print("  Sources:", df_legitimate['source'].value_counts().to_dict())

# Combine
df_all = pd.concat([df_malicious, df_legitimate], ignore_index=True)

# Remove duplicates
print(f"\nBefore deduplication: {len(df_all)}")
df_all = df_all.drop_duplicates(subset=['url'])
print(f"After deduplication: {len(df_all)}")

# Verify HTTPS
https_pct = df_all['url'].str.startswith('https://').mean()
print(f"\nHTTPS percentage: {https_pct:.1%}")

if https_pct < 0.95:
    print("⚠️  Warning: Some HTTP URLs slipped through, filtering...")
    df_all = df_all[df_all['url'].str.startswith('https://')]
    print(f"After HTTPS filter: {len(df_all)}")

# Check balance
mal_count = (df_all['label']==1).sum()
leg_count = (df_all['label']==0).sum()

print(f"\nFinal counts:")
print(f"  Malicious: {mal_count}")
print(f"  Legitimate: {leg_count}")
print(f"  Ratio: {mal_count/leg_count:.2f}")

# Shuffle
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
df_all.to_csv("urls_complete_https.csv", index=False)

print("\n" + "="*70)
print("✓ COMPLETE HTTPS DATASET SAVED")
print("="*70)
print(f"File: urls_complete_https.csv")
print(f"Total: {len(df_all)} URLs")
print(f"Malicious: {mal_count} (all HTTPS)")
print(f"Legitimate: {leg_count} (all HTTPS)")
print("\n✓ Ready to combine with your existing HTTPS dataset!")

print("\n" + "="*70)
print("NEXT STEP: COMBINE WITH EXISTING")
print("="*70)
print("Run this:")
print("""
import pandas as pd

df_existing = pd.read_csv('urls_master_https_only.csv')
df_new = pd.read_csv('urls_complete_https.csv')

df_combined = pd.concat([df_existing, df_new], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['url'])

print(f"Combined: {len(df_combined)} total URLs")
print(f"  Malicious: {(df_combined['label']==1).sum()}")
print(f"  Legitimate: {(df_combined['label']==0).sum()}")

df_combined.to_csv('urls_master_final.csv', index=False)
print("✓ Saved to urls_master_final.csv")
""")

COMPLETE HTTPS DATASET SCRAPER

PART 1: SCRAPING MALICIOUS URLS (HTTPS)

[1/6] Fetching OpenPhish feed...
  ✓ Added 115 phishing URLs

[2/6] Fetching PhishTank database...
  ✓ Added 8000 phishing URLs

[3/6] Fetching URLhaus malware feed...
  ✗ Failed: 'url_status'

PART 2: SCRAPING LEGITIMATE URLS (HTTPS)

[4/6] Fetching Tranco top domains...
  ✗ Failed: Error tokenizing data. C error: Expected 2 fields in line 5, saw 3


[5/6] Generating common web service URLs...
  ✓ Added 1988 web service URLs

[6/6] Generating Wikipedia URLs...
  ✓ Added 552 Wikipedia URLs

COMBINING RESULTS

Malicious URLs collected: 8115
  Sources: {'phishtank': 8000, 'openphish': 115}

Legitimate URLs collected: 2540
  Sources: {'web_services': 1988, 'wikipedia': 552}

Before deduplication: 10655
After deduplication: 9778

HTTPS percentage: 100.0%

Final counts:
  Malicious: 8111
  Legitimate: 1667
  Ratio: 4.87

✓ COMPLETE HTTPS DATASET SAVED
File: urls_complete_https.csv
Total: 9778 URLs
Malicious: 8111 (all 

: 

In [10]:
import pandas as pd

df_existing = pd.read_csv('urls_master_https_only.csv')
df_new = pd.read_csv('urls_complete_https.csv')

print("Existing:")
print(f"  Malicious: {(df_existing['label']==1).sum()}")
print(f"  Legitimate: {(df_existing['label']==0).sum()}")

print("\nNew:")
print(f"  Malicious: {(df_new['label']==1).sum()}")
print(f"  Legitimate: {(df_new['label']==0).sum()}")

df_combined = pd.concat([df_existing, df_new], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['url'])

print(f"\nCombined: {len(df_combined)} URLs")
print(f"  Malicious: {(df_combined['label']==1).sum()}")
print(f"  Legitimate: {(df_combined['label']==0).sum()}")

# Balance if needed
mal = (df_combined['label']==1).sum()
leg = (df_combined['label']==0).sum()

if mal > leg * 1.2:
    print(f"\nRebalancing (too many malicious)...")
    df_mal = df_combined[df_combined['label']==1].sample(n=leg, random_state=42)
    df_leg = df_combined[df_combined['label']==0]
    df_combined = pd.concat([df_mal, df_leg], ignore_index=True)
elif leg > mal * 1.2:
    print(f"\nRebalancing (too many legitimate)...")
    df_leg = df_combined[df_combined['label']==0].sample(n=mal, random_state=42)
    df_mal = df_combined[df_combined['label']==1]
    df_combined = pd.concat([df_mal, df_leg], ignore_index=True)

df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal: {len(df_combined)} URLs")
print(f"  Malicious: {(df_combined['label']==1).sum()}")
print(f"  Legitimate: {(df_combined['label']==0).sum()}")

df_combined.to_csv('urls_master_final.csv', index=False)
print("\n✓ Saved to urls_master_final.csv")
print("✓ Ready to extract features and train!")

Existing:
  Malicious: 3091
  Legitimate: 5493

New:
  Malicious: 8208
  Legitimate: 1666

Combined: 18440 URLs
  Malicious: 11299
  Legitimate: 7141

Rebalancing (too many malicious)...

Final: 14282 URLs
  Malicious: 7141
  Legitimate: 7141

✓ Saved to urls_master_final.csv
✓ Ready to extract features and train!
